# Create Expected Goals Table

In [2]:
from fotmob_api import FotmobAPI
import pandas as pd
from plottable.cmap import normed_cmap
from plottable import Table, ColumnDefinition
import matplotlib
import matplotlib.pyplot as plt

# Initialize client
client = FotmobAPI()

# Get all leagues
all_leagues = client.get_league_all()

# Create dataframe for country leagues
df_country_leagues = pd.json_normalize(all_leagues["countries"], record_path=["leagues"], meta=["ccode", "name", "localizedName"], record_prefix="league")
df_country_leagues.head()

,leagueid,leaguename,leaguelocalizedName,leaguepageUrl,leagueccode,ccode,name,localizedName
0,260,Kategoria Superiore,Kategoria Superiore,/leagues/260/overview/kategoria-superiore,ALB,ALB,Albania,Albania
1,9173,Superiore Qualification,Superiore Qualification,/leagues/9173/overview/superiore-qualification,ALB,ALB,Albania,Albania
2,10175,Superkupa e Shqipërisë,Superkupa e Shqipërisë,/leagues/10175/overview/superkupa-e-shqiperise,ALB,ALB,Albania,Albania
3,516,Ligue 1,Ligue 1,/leagues/516/overview/ligue-1,ALG,ALG,Algeria,Algeria
4,112,Liga Profesional,Liga Profesional,/leagues/112/overview/liga-profesional,ARG,ARG,Argentina,Argentina


In [ ]:
# Get Premier League league id
league_id = df_country_leagues[(df_country_leagues["leaguename"]=="Premier League") & (df_country_leagues["name"]=="England")]["leagueid"].item()

# Get table from Premier League
league_table = client.get_league_table(league_id=league_id)

table = league_table[0]["data"]["table"]
df_table_xg = pd.DataFrame(table["xg"])
df_table_xg.head()

In [ ]:
# Subset columns for table plot
columns = ["shortName", "played", "wins", "draws", "losses", "pts","xPoints",
           "xg","xgConceded", "xgDiff", "xPointsDiff", "idx"]
column_names = []
df_table_xg_plot = df_table_xg[columns].set_index("idx").round(1)

col_def = (
    [
        ColumnDefinition(
            name="idx",
            title="",
            textprops={"ha": "left"},
            width=0.5,
        ),
        ColumnDefinition(
            name="shortName",
            title="Team",
            textprops={"ha": "left", "weight": "bold"},
            width=1.5,
        ),
        ColumnDefinition(
            name="played",
            textprops={"ha": "center"},
            width=0.5,
            group="Basic"
        ),
        ColumnDefinition(
            name="wins",
            textprops={"ha": "center"},
            width=0.5,
            group="Basic"
        ),
        ColumnDefinition(
            name="draws",
            textprops={"ha": "center"},
            width=0.5,
            group="Basic"
        ),
        ColumnDefinition(
            name="losses",
            textprops={"ha": "center"},
            width=0.5,
            group="Basic"
        ),
        ColumnDefinition(
            name="pts",
            textprops={"ha": "center"},
            width=0.5,
            group="Basic",
            border="right"
        ),
        ColumnDefinition(
            name="xPoints",
            textprops={"ha": "center"},
            width=0.75,
            group="Expected Goals"
        ),
        ColumnDefinition(
            name="xg",
            textprops={"ha": "center"},
            width=0.75,
            group="Expected Goals"
        ),
        ColumnDefinition(
            name="xgConceded",
            textprops={"ha": "center"},
            width=0.75,
            group="Expected Goals"
        ),
        ColumnDefinition(
            name="xgDiff",
            textprops={"ha": "center"},
            width=0.75,
            group="Expected Goals"
        ),
        ColumnDefinition(
            name="xPointsDiff",
            textprops={"ha": "center"},
            width=0.75,
            group="Expected Goals",
            cmap=normed_cmap(df_table_xg_plot["xPointsDiff"], cmap=matplotlib.cm.coolwarm_r, num_stds=1.5)
        )
        ])

# Plot table
fig, ax = plt.subplots(figsize=(12, 12))
table_plot = Table(df=df_table_xg_plot, column_definitions=col_def, ax=ax, textprops={"fontsize": 12})

fig.suptitle("Premier League Standings", y=0.95, fontweight="bold", fontsize=24)
ax.set_title("Expected Goals Table - 2023/2024", fontweight="regular", fontsize=18)